# Customer Segmentation Analysis
# By Gunn Dumbwani 

**Track:** Data Analytics (OIBSIP) &nbsp;|&nbsp; **Level 1 - Task 2**


**Objective:** Segment an e-commerce company's customers into distinct behavioural groups using RFM 
(Recency, Frequency, Monetary) analysis + K-Means clustering, so the marketing team can target each 
group with a different strategy instead of a one-size-fits-all campaign.

**Dataset:** `Online_Retail_Transactions.csv` - a transactional export (InvoiceNo, StockCode, Quantity, 
InvoiceDate, UnitPrice, CustomerID, Country) covering ~900 customers over one year. It intentionally 
contains a few of the messy things real retail exports have: some cancelled orders (negative quantity, 
invoice numbers prefixed with `C`) and a handful of rows with a missing `CustomerID`.


## 1. Imports


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_style('whitegrid')
%matplotlib inline


ModuleNotFoundError: No module named 'sklearn'

## 2. Load & Inspect the Data


In [ ]:
df = pd.read_csv('Online_Retail_Transactions.csv', parse_dates=['InvoiceDate'])
print('Shape:', df.shape)
df.info()
print()
print('Nulls per column:')
print(df.isnull().sum())


145 rows have no `CustomerID` attached, so they can't be linked to any customer and will be dropped 
rather than imputed. The dataset also has cancelled-order rows (negative `Quantity`, invoice numbers 
starting with `C`) that need to come out too before we compute spend per customer.


## 3. Data Cleaning


In [ ]:
before = len(df)

# Drop rows with no customer attached - can't segment an unknown customer
df = df.dropna(subset=['CustomerID']).copy()
df['CustomerID'] = df['CustomerID'].astype(int)

# Remove cancellations/returns (negative quantity) - we only want completed purchases for RFM
df = df[df['Quantity'] > 0]

# Revenue per line item
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

after = len(df)
print(f'Rows before cleaning: {before}')
print(f'Rows after removing nulls & cancellations: {after}')
print(f'Rows dropped: {before - after}')


## 4. Feature Selection: RFM


For customer segmentation, three behavioural features capture almost everything that matters for 
marketing decisions:

- **Recency** - days since the customer's last order (lower = more engaged)
- **Frequency** - number of distinct orders placed (higher = more loyal)
- **Monetary** - total amount spent (higher = more valuable)

This is the classic **RFM** framework used across retail/e-commerce analytics.


In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

print(rfm.describe().round(2))


Recency ranges from 1 to 364 days, quite a wide spread, so the customer base has both very active 
buyers and customers who have effectively churned. Monetary is right-skewed too (mean £665 vs median 
£497), which is normal for spend data since a small number of big spenders pull the average up. That's 
why Frequency and Monetary get log-transformed before clustering, in the next section.


## 5. RFM Distributions


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
sns.histplot(rfm['Recency'], bins=30, ax=axes[0], color='#4C72B0')
axes[0].set_title('Recency (days since last order)')
sns.histplot(rfm['Frequency'], bins=30, ax=axes[1], color='#55A868')
axes[1].set_title('Frequency (distinct orders)')
sns.histplot(rfm['Monetary'], bins=30, ax=axes[2], color='#C44E52')
axes[2].set_title('Monetary (total spend)')
plt.tight_layout()
plt.show()


## 6. Feature Scaling


Frequency and Monetary are log-transformed first (`log1p`) to reduce the right-skew seen above, then 
all three RFM features are standardised with `StandardScaler` so that Monetary (which has much bigger 
raw values than Recency/Frequency) doesn't dominate the distance calculation K-Means relies on.


In [ ]:
rfm_log = rfm.copy()
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm_log['Monetary'])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log[['Recency', 'Frequency', 'Monetary']])
print('Scaled feature matrix shape:', rfm_scaled.shape)


## 7. Choosing K - the Elbow Method


In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(K_range), inertias, marker='o', color='#4C72B0')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia (within-cluster SSE)')
plt.title('Elbow Method for Optimal k')
plt.xticks(list(K_range))
plt.show()


Inertia drops sharply up to k=4 and then flattens out, adding a 5th or 6th cluster barely reduces 
error any further. So k=4 looks like the point of diminishing returns, and that's what I'll use for 
the final model.


## 8. Final Clustering (k=4)


In [ ]:
K_FINAL = 4
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
print(rfm['Cluster'].value_counts().sort_index())


## 9. Visualising the Clusters


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
palette = sns.color_palette('Set2', K_FINAL)

sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster',
                palette=palette, ax=axes[0], s=40, alpha=0.75)
axes[0].set_title('Recency vs Monetary by Cluster')

sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster',
                palette=palette, ax=axes[1], s=40, alpha=0.75)
axes[1].set_title('Frequency vs Monetary by Cluster')
plt.tight_layout()
plt.show()


## 10. Cluster Profiling


In [ ]:
profile = rfm.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(1)
print(profile)


**Naming the segments**, based on the profile above:

| Cluster | Label | Behaviour |
|---|---|---|
| 0 | **Champions** | Bought recently (~24 days), order very often (13x), highest spend (£1,361 avg) - the core high-value base |
| 1 | **New / One-off Buyers** | Very recent (~26 days) but only 1-2 orders and lowest spend - likely first-time or trial customers |
| 2 | **Occasional Regulars** | Moderate on all three metrics, buy every couple of months, mid-size baskets |
| 3 | **At Risk / Lapsed** | Haven't ordered in ~251 days on average, despite decent historical frequency (6.6) and spend (£674) - used to be good customers |


## 11. Customers per Segment


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x=profile.index.astype(str), y=profile['Customers'],
            hue=profile.index.astype(str), palette=palette, legend=False)
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Customers per Segment')
plt.show()


## 12. Marketing Recommendations per Segment

- **Champions (241 customers, 27%)** - Protect this revenue base. Give early access to new products 
and a loyalty/VIP tier. Don't blast them with generic discount emails - they're already converting; 
a small % here churning would hurt revenue disproportionately.

- **New / One-off Buyers (159 customers, 18%)** - Highest volume opportunity to convert into repeat 
customers. Trigger a "welcome back" email/discount 30-45 days after their first order, before they 
drift into the At-Risk group.

- **Occasional Regulars (322 customers, 36%, the largest group)** - Nudge order frequency up with 
reminder emails tied to their typical repurchase cycle, and cross-sell related products based on 
past orders.

- **At Risk / Lapsed (178 customers, 20%)** - These were previously decent-value customers (£674 avg 
spend, 6.6 orders) who've gone quiet for ~8 months. A win-back campaign (time-limited discount, 
"we miss you" email) is worth the cost here since their historical value is proven, cheaper to 
reactivate than to acquire a new customer from scratch.


## Conclusion

RFM + K-Means split the customer base into 4 clearly distinguishable segments (validated visually via 
the Recency-Monetary and Frequency-Monetary scatter plots, where clusters form separated regions rather 
than overlapping randomly). The most actionable finding is the At Risk segment (20% of customers), 
since they carry real historical value and are probably a better marketing investment than acquiring 
new customers, but they need a win-back trigger before they're lost for good.
